In [ ]:
!pip install -U google-genai sentence-transformers chromadb langchain-text-splitters pypdf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.8/739.8 kB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.8/393.8 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.4/262.4 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 57.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 36.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.

In [ ]:
import os
from google import genai
from google.colab import userdata

# Get API key from environment variable
api_key = os.environ.get("Gemini_API_Key")

# If not found, get it from Google Colab Secrets
if not api_key:
    try:
        api_key = userdata.get("Tejaswini")
    except Exception:
        api_key = None

if not api_key:
    raise ValueError("Gemini_API_Key not found. Add it to Colab Secrets.")

# Create Gemini client
client = genai.Client(api_key=api_key)

print("Gemini client initialized successfully.")

Gemini client initialized successfully.


In [ ]:
from google.colab import files
from pypdf import PdfReader

print("Please upload one or more PDF files:")

uploaded = files.upload()

pdf_texts = []

for filename in uploaded.keys():

    if filename.lower().endswith(".pdf"):

        reader = PdfReader(filename)

        text = ""

        for page_num, page in enumerate(reader.pages):

            page_text = page.extract_text()

            if page_text:
                text += f"\n--- Page {page_num + 1} ---\n"
                text += page_text

        pdf_texts.append(text)

        print(f"Loaded `{filename}` ({len(reader.pages)} pages.)")


if not pdf_texts:
    raise ValueError("No PDF files found in upload.")

full_pdf_content = "\n\n".join(pdf_texts)

print("PDF text extraction complete.")


Please upload one or more PDF files:


Saving GEN-AI 74 (1).pdf to GEN-AI 74 (1).pdf
Loaded `GEN-AI 74 (1).pdf` (7 pages.)
PDF text extraction complete.


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = text_splitter.split_text(full_pdf_content)

print(f"Extracted and split document into {len(chunks)} text chunks.")

Extracted and split document into 10 text chunks.


In [ ]:
print("Loading embedding model and building vector index...")

from sentence_transformers import SentenceTransformer
import chromadb

# Load embedding model
embedder = SentenceTransformer("all-MiniLM-L6-v2")

# Create ChromaDB client
chroma_client = chromadb.Client()

# Reset collection for clean execution
try:
    chroma_client.delete_collection(name="pdf_rag_collection")
except Exception:
    pass

# Create collection
collection = chroma_client.create_collection(
    name="pdf_rag_collection"
)

# Generate embeddings
chunk_embeddings = embedder.encode(chunks).tolist()

# Create unique IDs
chunks_ids = [
    f"doc_chunk_{i}"
    for i in range(len(chunks))
]

# Add documents and embeddings to ChromaDB
collection.add(
    documents=chunks,
    embeddings=chunk_embeddings,
    ids=chunks_ids
)

print("PDF Vector Indexing complete.")


Loading embedding model and building vector index...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

PDF Vector Indexing complete.
